<a href="https://colab.research.google.com/github/YTChiew/Marketing-Campaign-AB-Testing-Analysis/blob/main/campaign_analysis_pipeline.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# Notebook Title: Analysis
# Author: YTChiew
# Date: 2026-04-19
# Project: GitHub portfolio

#Includes:
  # - Data cleaning and preprocessing (EDA)
  # - SQL data loading and querying
  # - ANOVA statistical testing
  # - Tukey Honestly Significant Difference (HSD) Post HOC testing

In [2]:
#import libraries and dataset

import pandas as pd
import numpy as np

data = pd.read_csv("/content/WA_Marketing-Campaign.csv")

data.head()

,MarketID,MarketSize,LocationID,AgeOfStore,Promotion,week,SalesInThousands
0,1,Medium,1,4,3,1,33.73
1,1,Medium,1,4,3,2,35.67
2,1,Medium,1,4,3,3,29.03
3,1,Medium,1,4,3,4,39.25
4,1,Medium,2,5,2,1,27.81


In [3]:
data.info()

print("\n", "------------------------------------", "\n")

data.describe()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 548 entries, 0 to 547
Data columns (total 7 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   MarketID          548 non-null    int64  
 1   MarketSize        548 non-null    object 
 2   LocationID        548 non-null    int64  
 3   AgeOfStore        548 non-null    int64  
 4   Promotion         548 non-null    int64  
 5   week              548 non-null    int64  
 6   SalesInThousands  548 non-null    float64
dtypes: float64(1), int64(5), object(1)
memory usage: 30.1+ KB

 ------------------------------------ 



,MarketID,LocationID,AgeOfStore,Promotion,week,SalesInThousands
count,548.000000,548.000000,548.000000,548.000000,548.000000,548.000000
mean,5.715328,479.656934,8.503650,2.029197,2.500000,53.466204
std,2.877001,287.973679,6.638345,0.810729,1.119055,16.755216
min,1.000000,1.000000,1.000000,1.000000,1.000000,17.340000
25%,3.000000,216.000000,4.000000,1.000000,1.750000,42.545000
50%,6.000000,504.000000,7.000000,2.000000,2.500000,50.200000
75%,8.000000,708.000000,12.000000,3.000000,3.250000,60.477500
max,10.000000,920.000000,28.000000,3.000000,4.000000,99.650000


In [4]:
#check for null values

data.isnull().sum()

,0
MarketID,0
MarketSize,0
LocationID,0
AgeOfStore,0
Promotion,0
week,0
SalesInThousands,0


In [5]:
#any duplicated rows
data.duplicated().sum()

np.int64(0)

There are no dupicated rows or null values in the dataset


In [6]:
#load dataset into SQL

import sqlite3

conn = sqlite3.connect("marketing.db")

data.to_sql("campaign_data", conn, if_exists="replace", index=False)

548

In [7]:
pd.read_sql("SELECT * FROM campaign_data LIMIT 5", conn)

,MarketID,MarketSize,LocationID,AgeOfStore,Promotion,week,SalesInThousands
0,1,Medium,1,4,3,1,33.73
1,1,Medium,1,4,3,2,35.67
2,1,Medium,1,4,3,3,29.03
3,1,Medium,1,4,3,4,39.25
4,1,Medium,2,5,2,1,27.81


In [8]:
# Display Average sales per promotion:

pd.read_sql("""
SELECT
    Promotion,
    COUNT (*) AS total_records,
    AVG(SalesInThousands) AS avg_sales
FROM campaign_data
GROUP BY Promotion
ORDER BY avg_sales DESC
""", conn)

,Promotion,total_records,avg_sales
0,1,172,58.099012
1,3,188,55.364468
2,2,188,47.329415


Promotion 1 had the highest average sales (about 58.1k), surpassing Promotion 3 (about 55.4k) and significantly exceeding Promotion 2 (about 47.3k). The results show that Promotion 1 is the most effective campaign strategy, while Promotion 2 underperforms and may require improvisation or replacement.

In [9]:
# Disply Sales trend by week:

pd.read_sql("""
SELECT
    Promotion,
    AVG(SalesInThousands) AS avg_sales
FROM campaign_data
GROUP BY week, Promotion
ORDER BY week
""", conn)

,Promotion,avg_sales
0,1,58.244419
1,2,47.730213
2,3,55.776170
3,1,56.929535
4,2,47.582553
5,3,55.949149
6,1,58.774884
7,2,47.722128
8,3,54.377872
9,1,58.447209


Across the 4-week period, Promotion 1 consistently achieved the highest average sales, maintaining a stable performance range of approximately 57K to 59K. Promotion 3 followed with moderate performance (around 54K to 56K), while Promotion 2 consistently underperformed (around 46K to 48K).

The stability of these trends across all weeks suggests that Promotion 1 is
reliably the most effective strategy, whereas Promotion 2 shows persistently
lower performance and may require revision or replacement.

In [10]:
#Display Market size impact:

pd.read_sql("""
SELECT
    MarketSize,
    Promotion,
    AVG(SalesInThousands) AS avg_sales
FROM campaign_data
GROUP BY MarketSize, Promotion;
""", conn)

,MarketSize,Promotion,avg_sales
0,Large,1,75.235893
1,Large,2,60.322031
2,Large,3,77.203958
3,Medium,1,47.672604
4,Medium,2,39.114352
5,Medium,3,45.468879
6,Small,1,60.162500
7,Small,2,50.810625
8,Small,3,59.514167


Market size analysis reveals a distinct performance gradient. Large markets consistently achieve the highest average sales (approximately 60 to 70K), followed by small markets (around 50 to 60K), while medium markets significantly underperform (with only 39K to 47K).

This suggests that the market size plays an important factor in sales performance, with larger markets showing greater responsiveness to promotional campaigns. While promotion effectiveness is consistent across all segments, the persistently lower performance in medium-sized markets suggests potential structural or targeting issues that may require further investigation and optimisation.

**Statistical Testing: ANOVA**
---
A statistical method generally used to determine statistical differences between three or more groups.

H0 (null hypothesis): μ1 =μ2 =μ3 =… = μk
-> implies that the means of all the population are equal.

H1 (null hypothesis): It states that there will be at least one population mean that differs from the rest.

In [11]:
from scipy.stats import f_oneway

group1 = data[data['Promotion'] == 1]['SalesInThousands']
group2 = data[data['Promotion'] == 2]['SalesInThousands']
group3 = data[data['Promotion'] == 3]['SalesInThousands']

f_stat, p_value = f_oneway(group1, group2, group3)

print(f"F-statistic: {f_stat}", "\n")
print(f"P-value: {p_value}")


F-statistic: 21.953485793080674 

P-value: 6.765849261408714e-10


The output will provide two important values:

*F-statistic* : measures the variance between the groups relative to the variance within the groups.

*P-value* : helps determine if the results are statistically significant.


The ANOVA test produced a statistically significant result (p < 0.05), indicating that there are statistically meaningful difference in average sales across the three promotion groups.

Combined with the observed performance metrics, Promotion 1 consistently achieves the highest average sales, followed by Promotion 3, while Promotion 2 significantly underperforms.

This suggests that the choice of promotion strategy has a measurable impact on sales outcomes, and Promotion 1 should be prioritized for maximizing revenue.

**Post-hoc Testing**
---
Tukey Honestly Significant Difference (HSD)
- Since ANOVA only tells us that there’s a difference, but not which groups differ, Tukey’s HSD test will be used to test for pairwise comparisons between promotion types.

- pairwise_tukeyhsd: This function calculates all possible pairwise comparisons of group means to identify which pairs differ.
- endog=df['']: The endogenous (dependent) variable—numerical data to be compare.
- groups=df['']: The groups to be compared.
- alpha=0.05: The significance level. This sets a 5% risk of concluding that a difference exists when it actually doesn't (Type I error). Tukey's HSD adjusts for multiple comparisons, ensuring the family-wise error rate for all pairs combined is still 5%.


In [15]:
from statsmodels.stats.multicomp import pairwise_tukeyhsd

tukey_result = pairwise_tukeyhsd(
    endog=data['SalesInThousands'],
    groups=data['Promotion'],
    alpha=0.05
)

print(tukey_result)

 Multiple Comparison of Means - Tukey HSD, FWER=0.05 
group1 group2 meandiff p-adj   lower    upper  reject
-----------------------------------------------------
     1      2 -10.7696    0.0 -14.7738 -6.7654   True
     1      3  -2.7345 0.2444  -6.7388  1.2697  False
     2      3   8.0351    0.0   4.1208 11.9493   True
-----------------------------------------------------


Insights:
- Promotion 1 is much better than Promotion 2
- There is no significant difference between Promotion 1 and Promotion 3


The Tukey’s HSD test reveals that the difference between Promotion 1 and Promotion 3 is not statistically significant. However, Promotion 2 performs significantly worse than both.

This suggests that both Promotion 1 and Promotion 3 are viable strategies, while Promotion 2 should be de-prioritized or reconsidered.
